In [1]:
import os
import sys
import pandas as pd
import time

from ytmusicapi import YTMusic

# module_path = os.path.abspath(os.path.join('..'))
# if module_path not in sys.path: sys.path.append(module_path)
# from ytmusic_library import YTMusicPlaylists

import ytmusicapi as ytmusicapi
print(f'Using ytmusicapi version: {ytmusicapi.__version__}')

HEADER_FILE='../headers_auth.json'
print(f'Using header file: {HEADER_FILE}')
yt = ytmusicapi.YTMusic(HEADER_FILE)

# Y = YTMusicPlaylists(header=HEADER_FILE)
# yt_playlists = Y.playlists['title'].unique()
# print(f"Loaded {len(yt_playlists)} playlists.")

YTMUSIC_DB_TSV='../playlists/_tracks_db.tsv'
print(f'Using yt db file: {YTMUSIC_DB_TSV}')
yt_db = pd.read_csv(YTMUSIC_DB_TSV, sep='\t', index_col=0)
print(f"Loaded {len(yt_db)} ytmusic db entries.")


Using ytmusicapi version: 0.24.0
Using header file: ../headers_auth.json
Using yt db file: ../playlists/_tracks_db.tsv
Loaded 127173 ytmusic db entries.


In [2]:
VALID_IDS = set()
INVALID_IDS = set()

In [5]:
# Failed on index 20000:21000

PL_NAME = 'zzzz_tmp'
PL_DESC = 'for backlogging valid video ids'
PL_SIZE = 1000
START_AT = PL_SIZE
for i in range(START_AT, len(yt_db), PL_SIZE):
    try:
        pl_vids = list(yt_db.index[i-PL_SIZE:i])
        pl_id = yt.create_playlist(PL_NAME, PL_DESC, video_ids=pl_vids)
        # print(f'{PL_SIZE} tracks added to {PL_NAME} ({pl_id})')
        time.sleep(2)

        pl_res = yt.get_playlist(pl_id, limit=PL_SIZE)
        valid_ids = set()
        for t in pl_res['tracks']:
            valid_ids.add(t['videoId'])
        invalid_ids = set(pl_vids) - valid_ids
        VALID_IDS.update(valid_ids)
        INVALID_IDS.update(invalid_ids)
        perc_valid = round(100*len(VALID_IDS)/i)
        print(f'({i//PL_SIZE}/{len(yt_db)//PL_SIZE}): {len(invalid_ids)} are invalid tracks, {len(INVALID_IDS)} invalid ({perc_valid}% valid) so far')
        time.sleep(2)
        yt.delete_playlist(pl_id)
    except Exception as e:
        print(f'Failed on index {i-PL_SIZE}:{i}')
        print(e)




(1/127): 73 are invalid tracks, 73 invalid (94% valid) so far
(2/127): 77 are invalid tracks, 150 invalid (94% valid) so far
(3/127): 110 are invalid tracks, 260 invalid (93% valid) so far
(4/127): 111 are invalid tracks, 371 invalid (92% valid) so far
(5/127): 77 are invalid tracks, 448 invalid (92% valid) so far
(6/127): 218 are invalid tracks, 666 invalid (90% valid) so far
(7/127): 79 are invalid tracks, 745 invalid (91% valid) so far
(8/127): 123 are invalid tracks, 868 invalid (91% valid) so far
(9/127): 132 are invalid tracks, 1000 invalid (90% valid) so far
(10/127): 96 are invalid tracks, 1096 invalid (90% valid) so far
(11/127): 132 are invalid tracks, 1228 invalid (90% valid) so far
(12/127): 153 are invalid tracks, 1381 invalid (90% valid) so far
(13/127): 148 are invalid tracks, 1529 invalid (90% valid) so far
(14/127): 47 are invalid tracks, 1576 invalid (90% valid) so far
(15/127): 127 are invalid tracks, 1703 invalid (90% valid) so far
(16/127): 89 are invalid tracks, 1

In [6]:
yt_db.loc[yt_db.index.isin(VALID_IDS)].to_csv(
        '../_tracks_valid_db.tsv', header=True, index=True, sep='\t')
yt_db.loc[ yt_db.index.isin(INVALID_IDS)].to_csv(
        '../_tracks_invalid_db.tsv', header=True, index=True, sep='\t')


## v0

In [42]:
ALL_YT_IDS = set()
OK_YT_IDS = set()
NOT_OK_YT_IDS = set()
EXCEPTION_YT_IDS = set()

def save_tsvs(yt_db):
    yt_db.loc[
        yt_db.index.isin(ALL_YT_IDS)].to_csv(
            'yt_tracks_db_all.tsv', header=True, index=True, sep='\t')
    yt_db.loc[
        yt_db.index.isin(OK_YT_IDS2)].to_csv(
            'yt_tracks_db_ok.tsv', header=True, index=True, sep='\t')
    yt_db.loc[
        yt_db.index.isin(NOT_OK_YT_IDS)].to_csv(
            'yt_tracks_db_not_ok.tsv', header=True, index=True, sep='\t')
    yt_db.loc[
        yt_db.index.isin(EXCEPTION_YT_IDS)].to_csv(
            'yt_tracks_db_exception.tsv', header=True, index=True, sep='\t')

def load_tsvs():
    return [
        pd.read_csv('yt_tracks_db_all.tsv', index_col=0, sep='\t'),
        pd.read_csv('yt_tracks_db_ok.tsv',  index_col=0, sep='\t'),
        pd.read_csv('yt_tracks_db_not_ok.tsv', index_col=0, sep='\t'),
        pd.read_csv('yt_tracks_db_exception.tsv', index_col=0, sep='\t'),
    ]   

all_df, ok_df, notok_df, excpt_df = load_tsvs()

ALL_YT_IDS = set(all_df.index)
OK_YT_IDS = set(ok_df.index)
NOT_OK_YT_IDS = set(notok_df.index)
EXCEPTION_YT_IDS = set(excpt_df.index)
print(f'ok: {len(OK_YT_IDS)}\nnot ok: {len(NOT_OK_YT_IDS)}\nexception: {len(EXCEPTION_YT_IDS)}\nall: {len(ALL_YT_IDS)}')


ok: 5420
not ok: 692
exception: 261
all: 6373


In [40]:

for k in yt_db.index:
    if k in ALL_YT_IDS:
        continue
    ALL_YT_IDS.add(k)
    try:
        res = yt.get_song(k)
        time.sleep(1)
        if res['playabilityStatus']['status'] == 'OK':
            OK_YT_IDS.add(k)
        else:
            print(100*'-')
            print(f"Got not OK status:{res['playabilityStatus']}")
            NOT_OK_YT_IDS.add(k)
            entry = yt_db.loc[yt_db.index==k]
            print(entry[['title', 'artist', 'album']].T)
            # print(res)
    except Exception as e:
        EXCEPTION_YT_IDS.add(k)
        print(100*'X')
        print('\n\nERROR')
        print(e)
        print('\n\n')

save_tsvs(yt_db)


----------------------------------------------------------------------------------------------------
Got not OK status:{'status': 'UNPLAYABLE', 'reason': 'This video is not available', 'errorScreen': {'playerErrorMessageRenderer': {'reason': {'runs': [{'text': 'This video is not available'}]}, 'thumbnail': {'thumbnails': [{'url': '//s.ytimg.com/yts/img/meh7-vflGevej7.png', 'width': 140, 'height': 100}]}, 'icon': {'iconType': 'ERROR_OUTLINE'}}}, 'contextParams': 'Q0FFU0FnZ0I='}
                            0_ux4NyOl3E
title                        Tighten Up
artist         Archie Bell & The Drells
album   Dancing At The Nick At NiteClub
----------------------------------------------------------------------------------------------------
Got not OK status:{'status': 'UNPLAYABLE', 'reason': 'This video is not available', 'errorScreen': {'playerErrorMessageRenderer': {'reason': {'runs': [{'text': 'This video is not available'}]}, 'thumbnail': {'thumbnails': [{'url': '//s.ytimg.com/yts/img/meh

KeyboardInterrupt: 

In [41]:
save_tsvs(yt_db)
